In [6]:
import sys
import os

sys.path.append(os.path.abspath('../src'))

from test_connection import get_connection
import pandas as pd

engine = get_connection()

In [7]:
# Get the necessary data
# Use LEFT JOIN to not lose studies

query = """
SELECT
    s.nct_id,
    s.study_type,
    s.overall_status,
    s.phase,
    s.enrollment,
    s.has_dmc,
    s.is_fda_regulated_drug,
    s.start_date,
    sp.agency_class AS sponsor_type,
    cv.number_of_facilities,
    cv.actual_duration
FROM ctgov.studies s
LEFT JOIN ctgov.sponsors sp ON s.nct_id = sp.nct_id AND sp.lead_or_collaborator = 'lead'
LEFT JOIN ctgov.calculated_values cv ON s.nct_id = cv.nct_id
WHERE s.study_type = 'INTERVENTIONAL'
  AND s.overall_status IN ('COMPLETED', 'TERMINATED', 'WITHDRAWN', 'SUSPENDED')
"""

df = pd.read_sql(query, engine)

In [8]:
import numpy as np

# Shape and dtypes
print(f"Dataframe shape: {df.shape} rows, {df.shape[1]} columns\n")

# Audit dataframe
audit_data = []

for col in df.columns:
    missing = df[col].isnull().sum()
    missing_pct = (missing / len(df)) * 100
    unique_vals = df[col].nunique()
    dtype = df[col].dtype

    # fx
    examples = df[col].dropna().unique()[:3]

    audit_data.append({
        'Feature': col,
        'Data Type': dtype,
        'Missing Values': missing,
        'Missing %': f"{missing_pct:.2f}%",
        'Unique Values': unique_vals,
        'Examples': examples
    })

audit_df = pd.DataFrame(audit_data)
display(audit_df)

Dataframe shape: (288993, 11) rows, 11 columns



,Feature,Data Type,Missing Values,Missing %,Unique Values,Examples
0,nct_id,str,0,0.00%,288993,"[NCT02423551, NCT04063241, NCT01235559]"
1,study_type,str,0,0.00%,1,[INTERVENTIONAL]
2,overall_status,str,0,0.00%,4,"[COMPLETED, TERMINATED, WITHDRAWN]"
3,phase,str,107,0.04%,8,"[NA, PHASE3, PHASE1]"
4,enrollment,float64,3249,1.12%,5066,"[71.0, 264.0, 604.0]"
5,has_dmc,object,48290,16.71%,2,"[False, True]"
6,is_fda_regulated_drug,object,151569,52.45%,2,"[False, True]"
7,start_date,object,2546,0.88%,8145,"[2015-04-30, 2019-07-16, 2010-12-31]"
8,sponsor_type,str,0,0.00%,9,"[FED, OTHER, INDUSTRY]"
9,number_of_facilities,int64,0,0.00%,593,"[1, 90, 2]"


In [10]:
df_clean = df.copy()

# Drop unuseable columns
# study_type has zero variance since we only queried "Interventional
# actual_duration is data leakage and we will only use for EDA
cols_to_drop = ['study_type', 'actual_duration']
df_clean = df_clean.drop(columns=[col for col in cols_to_drop if col in df_clean.columns])

# Reveal hidden nulls
hidden_nulls = ['NA', 'N/A', 'Unknown', 'Not Applicable']
df_clean = df_clean.replace(hidden_nulls, np.nan)

# Convert dtype
df_clean['start_date'] = pd.to_datetime(df_clean['start_date'], errors='coerce')

# Run audit again
audit_data_clean = []

for col in df_clean.columns:
    missing = df_clean[col].isnull().sum()
    missing_pct = (missing / len(df_clean)) * 100
    unique_vals = df_clean[col].nunique()
    dtype = df_clean[col].dtype
    examples = df_clean[col].dropna().unique()[:3]

    audit_data_clean.append({
        'Feature': col,
        'Data Type': dtype,
        'Missing Values': missing,
        'Missing %': f"{missing_pct:.2f}%",
        'Unique Values': unique_vals,
        'Examples': examples
    })

audit_df_clean = pd.DataFrame(audit_data_clean)
display(audit_df_clean)

,Feature,Data Type,Missing Values,Missing %,Unique Values,Examples
0,nct_id,str,0,0.00%,288993,"[NCT02423551, NCT04063241, NCT01235559]"
1,overall_status,str,0,0.00%,4,"[COMPLETED, TERMINATED, WITHDRAWN]"
2,phase,str,136058,47.08%,7,"[PHASE3, PHASE1, PHASE4]"
3,enrollment,float64,3249,1.12%,5066,"[71.0, 264.0, 604.0]"
4,has_dmc,object,48290,16.71%,2,"[False, True]"
5,is_fda_regulated_drug,object,151569,52.45%,2,"[False, True]"
6,start_date,datetime64[s],2546,0.88%,8145,"[2015-04-30 00:00:00, 2019-07-16 00:00:00, 201..."
7,sponsor_type,str,0,0.00%,9,"[FED, OTHER, INDUSTRY]"
8,number_of_facilities,int64,0,0.00%,593,"[1, 90, 2]"


**Missing Data Mechanism Analysis**

Variables to explore: `phase`, `is_fda_regulated`, `has_dmc`, `enrollment`

In [13]:
import seaborn as sns
import matplotlib.pyplot as plt

# Missingness indicators
df_clean['phase_missing'] = df_clean['phase'].isnull()
df_clean['fda_missing'] = df_clean['is_fda_regulated_drug'].isnull()

# Hypothesis: they are missing on the SAME rows
crosstab_missing = pd.crosstab(
    index=df_clean['phase_missing'],
    columns=df_clean['fda_missing'],
    margins=True,
    margins_name="Total"
)

print("Crosstab of missingness indicators:")
display(crosstab_missing)

# Checking for pattern in sponsors related to phase
sponsor_phase_missing = pd.crosstab(
    index=df_clean['sponsor_type'],
    columns=df_clean['phase_missing'],
    normalize='index'
) * 100

print(f"\nPercentage of sponsors missing phase:")
display(sponsor_phase_missing.round(1).sort_values(by=True, ascending=False))

Crosstab of missingness indicators:


fda_missing,False,True,Total
phase_missing,,,
False,54064,98871,152935
True,83360,52698,136058
Total,137424,151569,288993



Percentage of sponsors missing phase:


phase_missing,False,True
sponsor_type,,
AMBIG,33.3,66.7
FED,33.9,66.1
OTHER_GOV,36.9,63.1
OTHER,37.6,62.4
INDIV,69.6,30.4
NETWORK,80.5,19.5
UNKNOWN,80.8,19.2
INDUSTRY,83.2,16.8
NIH,89.9,10.1


In [16]:
# Investigation of has_dmc
df_clean['dmc_missing'] = df_clean['has_dmc'].isnull()

# Crosstab (fill missing phases to get a better understanding)
phase_dmc_missing = pd.crosstab(
    index=df_clean['phase'].fillna('UNKNOWN_PHASE'),
    columns=df_clean['dmc_missing'],
    normalize='index'
) * 100

print("Percent missing has_dmc by phase:")
display(phase_dmc_missing.round(1).sort_values(by=True, ascending=False))

Percent missing has_dmc by phase:


dmc_missing,False,True
phase,,
PHASE3,75.9,24.1
PHASE1,77.6,22.4
PHASE2,79.4,20.6
PHASE4,82.4,17.6
PHASE2/PHASE3,83.0,17.0
PHASE1/PHASE2,84.0,16.0
UNKNOWN_PHASE,87.7,12.3
EARLY_PHASE1,88.1,11.9


In [17]:
# Investigation of enrollment
# status distribution for complete dataset
baseline_status = df_clean['overall_status'].value_counts(normalize=True) * 100

# status distribution only rows that is missing enrollment
missing_enrollment_status = df_clean[df_clean['enrollment'].isnull()]['overall_status'].value_counts(normalize=True) * 100

enrollment_comparison = pd.DataFrame({
    'Baseline': baseline_status,
    'Missing Enrollment': missing_enrollment_status
}).fillna(0).round(1)

display(enrollment_comparison)



,Baseline,Missing Enrollment
overall_status,,
COMPLETED,85.0,92.7
TERMINATED,9.9,6.6
WITHDRAWN,4.6,0.5
SUSPENDED,0.5,0.2


**Final Cleansing Pipeline**

In [20]:
def clean_and_impute_data(raw_df):
    df = raw_df.copy()

    print("Cleaning data...")

    # Drop unusable or leakage columns
    cols_to_drop = ['study_type', 'actual_duration']
    df = df.drop(columns=[col for col in cols_to_drop if col in df.columns])
    print(f"Dropped columns: {cols_to_drop}")

    # Standardize null values
    hidden_nulls = ['NA', 'N/A', 'Unknown', 'Not Applicable']
    df = df.replace(hidden_nulls, np.nan)
    print("Standardized null values to np.nan")

    # Handle dates
    if 'start_date' in df.columns:
        df['start_date'] = pd.to_datetime(df['start_date'], errors='coerce')
        # Extract year and drop original
        df['start_year'] = df['start_date'].dt.year
        df = df.drop(columns=['start_date'])
        print("Converted start_date to numerical 'start_year'")

    # Handle categorical MNAR
    cat_cols_to_fill = ['phase', 'is_fda_regulated_drug', 'has_dmc']
    for col in cat_cols_to_fill:
        if col in df.columns:
            df[col] = df[col].fillna('UNKNOWN')
            print(f"Filled missing values in {col}")

    print("Data cleaning pipeline complete!")
    return df

df_cleaned = clean_and_impute_data(df)

print("Missing values for the new dataset:")
print(df_model_ready.isnull().sum())

Cleaning data...
Dropped columns: ['study_type', 'actual_duration']
Standardized null values to np.nan
Converted start_date to numerical 'start_year'
Filled missing values in phase
Filled missing values in is_fda_regulated_drug
Filled missing values in has_dmc
Data cleaning pipeline complete!
Missing values for the new dataset:
nct_id                      0
overall_status              0
phase                       0
enrollment               3249
has_dmc                     0
is_fda_regulated_drug       0
sponsor_type                0
number_of_facilities        0
start_year               2546
dtype: int64


**Encoding of features**

In [22]:
# Map for target variable
target_mapping = {
    'COMPLETED': 1,
    'TERMINATED': 0,
    'WITHDRAWN': 0,
    'SUSPENDED': 0
}

df_cleaned['target'] = df_cleaned['overall_status'].map(target_mapping)
df_cleaned = df_cleaned.drop(columns=['overall_status'])
print("Converted 'overall_status' to binary target variable")

# One-Hot encoding of text/categorical features
categorical_cols = ['phase', 'sponsor_type', 'has_dmc', 'is_fda_regulated_drug']
df_encoded = pd.get_dummies(df_cleaned, columns=categorical_cols, drop_first=True)

# Make sure everything is (1/0)
for col in df_encoded.columns:
    if df_encoded[col].dtype == bool:
            df_encoded[col] = df_encoded[col].astype(int)

print(f"\nDataframe dimensions before encoding: {df_cleaned.shape}")
print(f"Dataframe dimensions after encoding: {df_encoded.shape}")

display(df_encoded.head())

Converted 'overall_status' to binary target variable

Dataframe dimensions before encoding: (288993, 9)
Dataframe dimensions after encoding: (288993, 24)


,nct_id,enrollment,number_of_facilities,start_year,target,phase_PHASE1,phase_PHASE1/PHASE2,phase_PHASE2,phase_PHASE2/PHASE3,phase_PHASE3,...,sponsor_type_INDUSTRY,sponsor_type_NETWORK,sponsor_type_NIH,sponsor_type_OTHER,sponsor_type_OTHER_GOV,sponsor_type_UNKNOWN,has_dmc_True,has_dmc_UNKNOWN,is_fda_regulated_drug_True,is_fda_regulated_drug_UNKNOWN
0,NCT02423551,71.0,1,2015.0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
1,NCT04063241,264.0,1,2019.0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
2,NCT01235559,604.0,90,2010.0,1,0,0,0,0,1,...,1,0,0,0,0,0,0,1,0,1
3,NCT04745767,54.0,2,2021.0,1,1,0,0,0,0,...,1,0,0,0,0,0,0,0,1,0
4,NCT00319332,0.0,8,2005.0,0,0,0,0,0,1,...,1,0,0,0,0,0,0,1,0,1


**MICE Imputation for `enrollment` and `start_year`**

In [33]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

print("Imputing missing values...")

# Exclude nct_id
# Exclude target to avoid data leakage
cols_to_exclude = ['nct_id', 'target']
df_features = df_encoded.drop(columns=cols_to_exclude)

# Impute
imputer = IterativeImputer(random_state=42, max_iter=10)
imputed_data = imputer.fit_transform(df_features)

# Recreate DataFrame with filled values
df_final = pd.DataFrame(imputed_data, columns=df_features.columns, index=df_features.index)
df_final['nct_id'] = df_encoded['nct_id']
df_final['target'] = df_encoded['target']

# To avoid decimal points for integer columns
df_final['enrollment'] = np.round(df_final['enrollment'])
df_final['start_year'] = np.round(df_final['start_year'])

# Guardrails for MICE imputations on start_year
df_final['enrollment'] = np.clip(df_final['enrollment'], 0, 1000000)
df_final['start_year'] = np.clip(df_final['start_year'], 1970, 2023)

print("Imputation complete!")
print(f"Missing values now: {df_final.isnull().sum().sum()}")
display(df_final[['start_year', 'enrollment']].describe())


Imputing missing values...
Imputation complete!
Missing values now: 0


,start_year,enrollment
count,288993.000000,288993.000000
mean,2014.125962,457.018346
std,6.383976,10345.985021
min,1970.000000,0.000000
25%,2010.000000,23.000000
50%,2015.000000,52.000000
75%,2019.000000,132.000000
max,2023.000000,1000000.000000


There are 0 typos in the raw data
